# 04 Reactor Pipeline EDA

Analysis of the construction, planned and proposed project pipeline: size, technology, geography, timing, and risk scores.

> Run the pipeline first: `python run_pipeline.py`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = ROOT / 'data' / 'processed'
predictions = ROOT / 'outputs' / 'predictions'

pipeline = pd.read_csv(processed / 'reactor_pipeline.csv')
taxonomy = pd.read_csv(processed / 'technology_taxonomy.csv')
print(f"Pipeline projects: {len(pipeline)}")
pipeline.groupby('status_group').agg(projects=('project_id','count'), capacity_gwe=('capacity_mwe', lambda x: round(x.sum()/1000,1)))

## Pipeline funnel by status and technology

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

status_cap = pipeline.groupby('status_group')['capacity_mwe'].sum().div(1000).sort_values(ascending=False).reset_index()
sns.barplot(data=status_cap, x='status_group', y='capacity_mwe', ax=axes[0], palette='Blues_d')
axes[0].set_title('Pipeline Capacity by Status (GWe)'); axes[0].set_xlabel(''); axes[0].set_ylabel('GWe')

tech_cap = pipeline.groupby('technology_family')['capacity_mwe'].sum().div(1000).sort_values(ascending=False).reset_index()
sns.barplot(data=tech_cap, x='technology_family', y='capacity_mwe', ax=axes[1], palette='Greens_d')
axes[1].set_title('Pipeline Capacity by Technology (GWe)')
axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

## Expected additions timeline

In [ ]:
timeline = pipeline.groupby('expected_operation_year')['capacity_mwe'].sum().div(1000).reset_index()
timeline.columns = ['year','capacity_gwe']

fig, ax = plt.subplots(figsize=(11,5))
sns.barplot(data=timeline, x='year', y='capacity_gwe', ax=ax, color='steelblue')
ax.set_title('Expected Pipeline Additions by Year (GWe)', fontsize=13)
ax.set_ylabel('GWe'); ax.set_xlabel('Expected operation year')
plt.tight_layout(); plt.show()

**Important caveat:** `expected_operation_year` is a heuristic estimate based on status group (Construction → 2030, Planned → 2035, Proposed → 2042). Real project timelines vary significantly.

## Project maturity vs delay risk matrix

In [ ]:
fig = px.scatter(
    pipeline,
    x='project_maturity_score', y='delay_risk_score',
    size='capacity_mwe', color='status_group',
    hover_name='reactor_name',
    hover_data=['country','technology_family','capacity_mwe'],
    title='Project Maturity vs Delay Risk (bubble = capacity)',
    template='plotly_white'
)
fig.show()

**Reading the matrix:** Construction-stage projects cluster top-left (high maturity, lower risk). Proposed projects cluster bottom-right. Large first-of-a-kind units get a delay penalty regardless of status — reflecting historical evidence on construction complexity.

## Realization probability distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

sns.histplot(pipeline['realization_probability'], bins=15, ax=axes[0], color='teal')
axes[0].set_title('Realization Probability Distribution')
axes[0].set_xlabel('Probability')
axes[0].axvline(0.6, color='red', linestyle='--', label='High/Low threshold (0.6)')
axes[0].legend()

real_country = pipeline.groupby('country')['realization_probability'].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=real_country, x='country', y='realization_probability', ax=axes[1], palette='viridis')
axes[1].set_title('Avg. Realization Probability by Country')
axes[1].set_xlabel(''); axes[1].set_ylabel('Probability')
axes[1].tick_params(axis='x', rotation=35)
axes[1].axhline(0.6, color='red', linestyle='--', alpha=0.5)

plt.tight_layout(); plt.show()

## Technology mix in pipeline vs operating fleet

In [ ]:
reactors = pd.read_csv(processed / 'reactors_master.csv')
operating_tech = reactors[reactors.status_group=='Operating'].groupby('technology_family')['capacity_mwe'].sum().div(1000)
pipeline_tech = pipeline.groupby('technology_family')['capacity_mwe'].sum().div(1000)

compare = pd.DataFrame({'Operating (GWe)': operating_tech, 'Pipeline (GWe)': pipeline_tech}).fillna(0)
compare.plot.bar(figsize=(11,5), colormap='tab10')
plt.title('Technology Mix: Operating Fleet vs Pipeline', fontsize=13)
plt.ylabel('GWe'); plt.xlabel('')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

**Shift toward advanced technologies.** The pipeline has a higher share of SMR and Gen IV projects relative to the operating fleet. Whether this translates to commissioned capacity depends heavily on financing, licensing, and supply chain development.